# T1 · Línea base de T-DEED sobre metraje propio

**Qué contesta este notebook**: si el modelo de *ball action spotting* publicado por
T-DEED ve algo en vídeo de nuestra cámara. No reproduce el paper ni mide contra SoccerNet:
mide contra lo nuestro, que es la pregunta que decide si este camino sigue.

**No hace falta el dataset de 19 GB.** Solo los pesos y un vídeo.

**Sin ejecutar todavía**: esto se escribió sin GPU delante. Si algo falla, lo que falle y
cómo se arregló va a `docs/PROGRESS.md` del repo, que para eso está.

---

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno → GPU**.

In [ ]:
!nvidia-smi
import torch

print("torch", torch.__version__, "· cuda", torch.cuda.is_available())

## 1. T-DEED

Se clona en cada sesión: son unos megabytes. **Es GPL-3.0** — se ejecuta, no se
redistribuye, y nada suyo entra en el producto sin resolver antes la licencia
(`docs/DEPENDENCIES.md` del repo de entrenamiento).

In [ ]:
%cd /content
!rm -rf T-DEED
!git clone --depth 1 https://github.com/arturxe2/T-DEED
%cd /content/T-DEED
!git rev-parse --short HEAD

## 2. Dependencias

**No instales su `requirements.txt`.** Pinea `torch==2.3.1` y `numpy==1.26.4` y se pelea
con lo que Colab trae hoy; reinstalar torch cuesta diez minutos y suele romper CUDA.
Solo lo que falta de verdad:

In [ ]:
!pip -q install timm tabulate wandb
import os

# `inference.py` importa wandb y no lo usa para nada en este camino.
os.environ["WANDB_MODE"] = "disabled"

## 3. Los pesos

`inference.py` los busca en una ruta exacta y **no es la obvia**:

```
checkpoints/SoccerNetBall/SoccerNetBall_challenge1/checkpoint_best.pt
                ^^^^^^^^^^^^^^  el prefijo del nombre, otra vez
```

Ponerlos un nivel más arriba —que es lo que haría cualquiera mirando la carpeta— falla
con un `FileNotFoundError` a mitad de la carga.

Elige **una** de las dos celdas según dónde los tengas.

In [ ]:
# --- opción A: Google Drive ---
from google.colab import drive

drive.mount("/content/drive")

PESOS_ORIGEN = "/content/drive/MyDrive/tdeed/checkpoint_best.pt"  # <-- ajusta

In [ ]:
# --- opción B: Google Cloud Storage ---
from google.colab import auth

auth.authenticate_user()
!gsutil cp gs://TU-BUCKET/tdeed/checkpoint_best.pt /content/checkpoint_best.pt

PESOS_ORIGEN = "/content/checkpoint_best.pt"

In [ ]:
import shutil
from pathlib import Path

MODELO = "SoccerNetBall_challenge1"
destino = Path("checkpoints") / MODELO.split("_")[0] / MODELO / "checkpoint_best.pt"
destino.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(PESOS_ORIGEN, destino)
print(destino, f"{destino.stat().st_size / 1e6:.0f} MB")

## 4. El vídeo, a 25 fps

**Esta es la trampa que más caro sale.** `ActionSpotInferenceDataset` lee el vídeo con
OpenCV y se queda con **uno de cada dos frames nativos**. No remuestrea a ninguna
frecuencia: aplica el stride a lo que le des.

El modelo se entrenó con frames extraídos a 25 fps y stride 2, o sea **12,5 fps**. Si le
das un vídeo a 30 fps, el clip le llega a 15 fps y las ocho segundos de ventana pasan a
ser seis y medio: el modelo ve la jugada acelerada respecto a todo lo que aprendió. No da
ningún error; solo acierta menos, y te quedas pensando que el modelo no sirve.

Así que se convierte a 25 fps primero. De paso se reescala a 796×448 —el dataset lo haría
igualmente— porque decodificar 4K frame a frame en Colab es lentísimo.

In [ ]:
VIDEO_ORIGEN = "/content/drive/MyDrive/videoGP.MP4"  # <-- ajusta
VIDEO = "/content/entrada_25fps.mp4"

!ffprobe -v error -select_streams v:0 -show_entries stream=r_frame_rate,width,height -of default=nw=1 "$VIDEO_ORIGEN"
!ffmpeg -y -v error -i "$VIDEO_ORIGEN" -r 25 -vf scale=796:448 -an "$VIDEO"
!ffprobe -v error -select_streams v:0 -show_entries stream=r_frame_rate,nb_frames -of default=nw=1 "$VIDEO"

## 5. Inferencia

El umbral bajo (0.2, su valor por defecto) es a propósito: lo que interesa ahora es ver
**si aparece algo**, no tener pocos falsos positivos. Afinar el umbral viene después, con
precisión y recall medidos.

In [ ]:
%cd /content/T-DEED
!python inference.py     --model SoccerNetBall_challenge1     --video_path "$VIDEO"     --frame_width 796 --frame_height 448     --inference_threshold 0.2

## 6. Qué encontró

La salida cae en `inference_output/results_inference.json`, con el número de frame
**nativo** (ya multiplicado por el stride). A 25 fps, segundos = frame / 25.

In [ ]:
import json
from collections import Counter

FPS = 25
datos = json.load(open("inference_output/results_inference.json"))
eventos = datos["predictions"]
print(f"{len(eventos)} eventos por encima del umbral")
print()

print("por clase:")
for clase, cuantos in Counter(e["label"] for e in eventos).most_common():
    print(f"  {clase:<26} {cuantos}")

In [ ]:
# Las tres que nos importan del MVP. El córner no está en esta tarea (ADR 0001).
NUESTRAS = {"GOAL", "SHOT", "FREE KICK"}


def mmss(segundos):
    return f"{int(segundos) // 60:02d}:{segundos % 60:05.2f}"


filtrados = sorted(
    (e for e in eventos if e["label"] in NUESTRAS),
    key=lambda e: -e["confidence"],
)
print(f"{'min:seg':>9}  {'clase':<12} confianza")
for e in filtrados[:40]:
    print(f"{mmss(e['frame'] / FPS):>9}  {e['label']:<12} {e['confidence']:.3f}")

## 7. Lo que hay que apuntar

Ve al vídeo, mira **tres o cuatro** de los instantes de arriba y contesta a esto en
`docs/PROGRESS.md`:

1. ¿Coincide un `GOAL` de confianza alta con un gol de verdad? ¿Con cuántos segundos de
   error?
2. ¿Cuántos `SHOT` hay y cuántos son tiros de verdad? Con el umbral en 0.2 habrá ruido;
   lo que interesa es si el ruido es *ruido* o es sistemático.
3. ¿Sale algo a 0.9 que no sea nada? Un falso positivo muy seguro es peor señal que
   muchos falsos positivos dudosos.
4. Cuánto tardó la inferencia y en qué GPU.

**Con eso se decide si seguir.** Si sobre metraje propio no aparece ni un gol, el problema
no es el umbral: es que el modelo mira transmisiones de TV y nuestra cámara es otra cosa,
y entonces el camino es fine-tuning con datos propios (T4) o cambiar de arquitectura.

Y si sale bien, lo siguiente es T5: exportar a ONNX **desde aquí**, donde vive torch, con
su ficha para `models/registry.yaml`.